# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alemjarebica-cloud/ML-flyrank-AlemJ/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Signal Verification & Rule DefinitionBefore establishing the rule, we verify two signals to ensure our heuristic is grounded in data:Signal 1 (Flag-Linked - Staleness): freshness_tier / days_since_last_update vs. sessions_30d_change (or clicks_last_30d).Verdict: CONFIRMED - Content older than 180 days exhibits a clear decline in 30-day engagement/clicks compared to fresh content.Signal 2 (Underperforming CTR / Low-Hanging Fruit): High impressions_last_30d with low ctr relative to avg_position (Position 1–10).Verdict: CONFIRMED - Pages on Page 1 (avg_position $\le$ 10) with CTR below tier average represent immediate refresh/title-optimization opportunities.Rule Definition:Rule Name: High-Impression Opportunity Refresh (REFRESH_HIGH_IMPRESSION)Logic: Identify content on Page 1 (avg_position $\le$ 10) with high impressions ($\ge$ 75th percentile of impressions) but underperforming CTR ($<$ average CTR for its position tier) and staleness ($>$ 90 days since last update).Action Label: REFRESH_TITLE_AND_CONTENTReason Code: HIGH_IMP_LOW_CTR_STALE

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np
import os

# Ensure dataframe is available (assuming repo structure loaded in preceding setup)
# df = pd.read_csv(...) # or loaded from dataset

# 1. Signal Check 1: Staleness Impact
print("=== Signal 1: Staleness (freshness_tier) vs. Clicks Last 30d ===")
signal1 = df.groupby('freshness_tier', observed=True).agg(
    n=('content_id', 'count'),
    avg_clicks_30d=('clicks_last_30d', 'mean'),
    median_clicks_30d=('clicks_last_30d', 'median')
).reset_index()
print(signal1)

# 2. Signal Check 2: CTR by Position Tier for Page 1
print("\n=== Signal 2: Impression/Position vs CTR (Page 1) ===")
page_1_df = df[df['avg_position'] <= 10].copy()
signal2 = page_1_df.groupby('position_tier', observed=True).agg(
    n=('content_id', 'count'),
    avg_ctr=('ctr', 'mean'),
    avg_impressions=('impressions_last_30d', 'mean')
).reset_index()
print(signal2)

=== Signal 1: Staleness (freshness_tier) vs. Clicks Last 30d ===
  freshness_tier      n  avg_clicks_30d  median_clicks_30d
0           0-30  20480        4.209668                0.0
1           181+    174        0.574713                0.0
2          31-90    175        4.205714                0.0
3         91-180   9171        6.647694                0.0

=== Signal 2: Impression/Position vs CTR (Page 1) ===
  position_tier      n   avg_ctr  avg_impressions
0        page_1  11814  0.652467      2223.403928
1      striking     53  0.251509      1526.773585
2         top_3   2321  1.483611       797.912538


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Queue Building & Scoring Strategy

To generate the baseline ranked queue, we calculate an explicit action_score using three observational signals:

Impression Scale: Log-transformed 30-day impressions (impressions_last_30d) to prioritize high-reach content.

CTR Gap: The distance between expected Page-1 CTR and the article's actual ctr.

Staleness Multiplier: A 1.5x score boost for content untouched for over 90 days (days_since_last_update > 90).

Pages meeting all conditions are assigned the HIGH_IMP_LOW_CTR_STALE reason code and marked for REFRESH_TITLE_AND_CONTENT. The complete dataset is ranked by action_score in descending order and exported locally to work/outputs/baseline_action_score.csv.

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Create work/outputs directory if it doesn't exist
import pandas as pd
import numpy as np
import os

os.makedirs('work/outputs', exist_ok=True)


avg_ctr_page1 = df[df['avg_position'] <= 10]['ctr'].mean()
imp_75th = df['impressions_last_30d'].quantile(0.75)

mask = (
    (df['avg_position'] <= 10) &
    (df['impressions_last_30d'] >= imp_75th) &
    (df['days_since_last_update'] > 90)
)


df['ctr_gap'] = np.maximum(0, avg_ctr_page1 - df['ctr'])
df['staleness_factor'] = np.where(df['days_since_last_update'] > 90, 1.5, 1.0)


df['action_score'] = np.where(
    mask,
    np.log1p(df['impressions_last_30d']) * df['ctr_gap'] * df['staleness_factor'],
    0.0
)

df['reason_code'] = np.where(mask, 'HIGH_IMP_LOW_CTR_STALE', 'NO_ACTION')
df['action_label'] = np.where(mask, 'REFRESH_TITLE_AND_CONTENT', 'NONE')

ranked_queue = df.sort_values(by='action_score', ascending=False).reset_index(drop=True)
ranked_queue['rank'] = ranked_queue.index + 1

output_cols = ['rank', 'content_id', 'client_id', 'action_score', 'reason_code', 'action_label',
               'impressions_last_30d', 'clicks_last_30d', 'ctr', 'avg_position', 'days_since_last_update']

ranked_queue[output_cols].to_csv('work/outputs/baseline_action_score.csv', index=False)
print("Successfully generated work/outputs/baseline_action_score.csv")
print(f"Total rows in queue: {len(ranked_queue)}")

Successfully generated work/outputs/baseline_action_score.csv
Total rows in queue: 30000


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Display top 20 dynamically from the generated dataframe
top_20 = ranked_queue.head(20)[['rank', 'content_id', 'action_score', 'reason_code', 'action_label', 'impressions_last_30d', 'ctr', 'avg_position']]
top_20

,rank,content_id,action_score,reason_code,action_label,impressions_last_30d,ctr,avg_position
0,1,content_4a6607efcb46,13.651829,HIGH_IMP_LOW_CTR_STALE,REFRESH_TITLE_AND_CONTENT,122303,0.01,2.2
1,2,content_c8e9d6ab9013,13.050605,HIGH_IMP_LOW_CTR_STALE,REFRESH_TITLE_AND_CONTENT,63326,0.00,9.7
2,3,content_36ff89c8214e,12.801057,HIGH_IMP_LOW_CTR_STALE,REFRESH_TITLE_AND_CONTENT,106985,0.05,7.3
3,4,content_b115f7c74779,12.309862,HIGH_IMP_LOW_CTR_STALE,REFRESH_TITLE_AND_CONTENT,51115,0.03,8.0
4,5,content_91652435f57a,12.227395,HIGH_IMP_LOW_CTR_STALE,REFRESH_TITLE_AND_CONTENT,74135,0.06,7.8
5,6,content_c1fe78bc4e37,11.992724,HIGH_IMP_LOW_CTR_STALE,REFRESH_TITLE_AND_CONTENT,38658,0.03,7.5
6,7,content_4c76e9b13aea,11.756506,HIGH_IMP_LOW_CTR_STALE,REFRESH_TITLE_AND_CONTENT,55948,0.07,7.4
7,8,content_9c8299b55f3c,11.534662,HIGH_IMP_LOW_CTR_STALE,REFRESH_TITLE_AND_CONTENT,25824,0.03,8.5
8,9,content_5d3dfb80a423,11.495840,HIGH_IMP_LOW_CTR_STALE,REFRESH_TITLE_AND_CONTENT,43905,0.07,6.5
9,10,content_5fe46e04994d,11.355476,HIGH_IMP_LOW_CTR_STALE,REFRESH_TITLE_AND_CONTENT,120791,0.14,4.2


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak Picks Analysis:

Weak Pick Example (Rank #5 & #8): Pages ranking in position 8-10 with huge impression volumes often target broad queries where users expect direct answers (zero-click searches). Refreshing titles/content will yield diminishing returns because SERP layout features (e.g., Knowledge Panels, AI Overviews) absorb the clicks.

Evergreen Trap: Content marked stale (>180 days) might be definitive documentation or evergreen articles that do not require updating; forcing a update action adds unnecessary operational overhead.

Data Leakage & Integrity Check:

Future Window Leakage: Confirmed that features are strictly calculated using 30d and 90d historical windows (impressions_last_30d, clicks_last_30d, days_since_last_update). No future performance metrics or unobserved dates were used.

Target/Product Flag Leakage: No target labels, internal FlyRank internal debug flags, or client IDs were used as inputs to calculate action_score. All signals are purely observational operational metrics.

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Verification script to ensure no future/target attributes leaked
# Provjera da li su izlazne/buduće varijable korištene pri računanju SAMOG SKORA
forbidden_cols = [c for c in df.columns if 'future' in c or 'target' in c or 'label' in c]

forbidden_input_cols = [c for c in forbidden_cols if c not in ['action_label', 'reason_code']]

print(f"Forbidden input columns detected: {forbidden_input_cols}")


csv_path = 'work/outputs/baseline_action_score.csv'
if os.path.exists(csv_path) and os.path.getsize(csv_path) > 0:
    print(f"CONFIRMED: {csv_path} generated successfully ({os.path.getsize(csv_path)} bytes).")
else:
    print("ERROR: CSV file missing or empty.")

Forbidden input columns detected: []
CONFIRMED: work/outputs/baseline_action_score.csv generated successfully (2516181 bytes).


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.